# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '/content/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# find poptart rows
pop_rows = cat[cat['product_name'].str.lower().str.contains('pop')]

# display details
print("poptart rows: ",len(pop_rows))
display(pop_rows)

poptart rows:  4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# strawberry poptart
strawberry_pop = pop_rows[pop_rows['product_name'].str.lower().str.contains("strawberry")]

# non strawberry poptart
non_strawberry = pop_rows[~pop_rows['product_name'].str.lower().str.contains("strawberry")]

# display them
display(strawberry_pop[['sku', 'product_name']])
display(non_strawberry[['sku', 'product_name']])

,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry
3,PT_STRAWBERRY,pop-tarts strawberry


,sku,product_name


**Pop-Tart rows that are a different product:** _..._

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# first map all strawberry SKU to one SKU
CANON = {
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW',
}

# consolidation function similar to the studio
def consolidate(df):
  out = df.copy()
  out['canonical_sku'] = out['sku'].map(CANON).fillna(out['sku'])
  return out

# consolidate the original set
consolidated = consolidate(cat)

# check that it worked
display(consolidated[consolidated['sku'].isin(CANON)][['sku', 'product_name', 'canonical_sku']])

,sku,product_name,canonical_sku
0,PT-12,pop-tarts strawberry,POPTART-STRAW
1,SKU#459812,Strawberry Pop-Tart,POPTART-STRAW
2,POPTART-STRAW,Pop-Tarts Strawberry,POPTART-STRAW
3,PT_STRAWBERRY,pop-tarts strawberry,POPTART-STRAW


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# show the before
print("Before")
print(strawberry_pop['category'].value_counts())

# create a copy of the original dataset
strawberry_clean = strawberry_pop.copy()

# remove extra spacing and standardize capitalization
strawberry_clean['category'] = (strawberry_clean['category'].str.strip().str.title())

# show the after
print("\nAfter")
print(strawberry_clean['category'].value_counts())



Before
category
Breakfast          2
Breakfast Foods    1
breakfast          1
Name: count, dtype: int64

After
category
Breakfast          3
Breakfast Foods    1
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# Normalize all of the product names
candidates = cat.copy()
candidates['name_key'] = (candidates['product_name'].str.lower().str.replace(r'[^a-z0-9\s]', ' ', regex=True).str.split().apply(lambda words: ' '.join(sorted(words))))

# keep the groups where the normalized name has more than one SKU
fragmented = candidates.groupby('name_key').filter(lambda group: group['sku'].nunique() > 1)

display(
    fragmented[['name_key', 'sku', 'product_name']]
    .sort_values('name_key')
)

,name_key,sku,product_name
12,24 beer light pack,BEER-24,Light Beer (24-pack)
13,24 beer light pack,LTBEER24,Light Beer (24-pack)
8,4 aa batteries pack,BAT-AA4,AA Batteries 4-Pack
9,4 aa batteries pack,BATT_AA_4PK,AA Batteries 4-Pack
30,aid first kit,FIRSTAID-KIT,First Aid Kit
29,aid first kit,FAID-01,First-Aid Kit
15,board game monopoly,MONOPOLY-01,Monopoly Board Game
14,board game monopoly,BG-MONO,Monopoly Board Game
16,board game monopoly,GAME-MONO,Board Game - Monopoly
24,bread white,WHITEBREAD,WHITE BREAD


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
# Check that consolidation did not lose any data
assert consolidated['canonical_sku'].notna().all(), \
    'canonical SKU contains nulls'

assert len(consolidated) == len(cat), \
    'consolidation changed the number of rows'

print('mapping is safe')

mapping is safe


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

**a)** The heavy duty tarp is one example that appeared fragmented, as there were some labeled TARP - HEAVY DUTY and some labeled Heavy Duty Tarp. These got grouped using "duty heavy tarp". Another example of fragmented products were first aid kits, where some were named First Aid Kit and some were named First-Aid Kit. These were grouped using "aid first kit".

**b)** Consolidation stops being a cleaning task when it becomes unclear whether two similar names refer to the same product or not. One example from this catalog where the decision to consolodate becomes confusing is with variety packs for chips. Often there are different chip variety pack versions or sizes, so different names could indicate different chip assortments or sizes, and may not want to be grouped together. This is something you could ask a category manager or buyer about who knows the products before consolidating them.